# Imports

In [1]:
import torch
from torchvision import transforms
from torchvision.models import resnet18, resnet50, resnet101
from torchvision.models import ResNet18_Weights, ResNet50_Weights, ResNet101_Weights
import requests
from PIL import Image
from io import BytesIO
import time

# Preprocess

In [2]:
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# Labels

In [3]:
labels_url = "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt"
labels = requests.get(labels_url).text.split("\n")

# Models

In [4]:
models_dict = {
    "ResNet18": resnet18(weights=ResNet18_Weights.DEFAULT).eval(),
    "ResNet50": resnet50(weights=ResNet50_Weights.DEFAULT).eval(),
    "ResNet101": resnet101(weights=ResNet101_Weights.DEFAULT).eval()
}

# Uploading Images

In [5]:
def load_image(url):
    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(url, headers=headers)

    if response.status_code != 200:
        raise Exception(f"Error loading image: {response.status_code}")

    return Image.open(BytesIO(response.content)).convert("RGB")

# Classification

In [6]:
def classify_image(model, url, top_k=5):
    img = load_image(url)
    img_tensor = preprocess(img).unsqueeze(0)

    start = time.time()

    with torch.no_grad():
        output = model(img_tensor)

    inference_time = time.time() - start

    probs = torch.nn.functional.softmax(output[0], dim=0)
    top_probs, top_idxs = torch.topk(probs, top_k)

    results = []
    for prob, idx in zip(top_probs, top_idxs):
        results.append({
            "class": labels[idx.item()],
            "prob": prob.item()
        })

    return results, inference_time

# Data

In [7]:
image_urls = [
    "https://images.unsplash.com/photo-1517849845537-4d257902454a",  # dog
    "https://images.unsplash.com/photo-1518791841217-8f162f1e1131",  # cat
    "https://images.unsplash.com/photo-1502877338535-766e1452684a",  # car
    "https://images.unsplash.com/photo-1517336714731-489689fd1ca8",  # laptop
    "https://images.unsplash.com/photo-1493238792000-8113da705763"   # airplane
]

# TASK 1 + 2

In [8]:
for url in image_urls:
    print("\nImage:", url)

    results, t = classify_image(models_dict["ResNet50"], url)

    print(f"Inference time: {t:.4f} sec")

    for i, r in enumerate(results, 1):
        print(f"{i}. {r['class']} - {r['prob']*100:.2f}%")


Image: https://images.unsplash.com/photo-1517849845537-4d257902454a
Inference time: 0.0953 sec
1. pug - 39.17%
2. Brabancon griffon - 2.01%
3. French bulldog - 0.90%
4. tennis ball - 0.49%
5. bull mastiff - 0.37%

Image: https://images.unsplash.com/photo-1518791841217-8f162f1e1131
Inference time: 0.1188 sec
1. tiger cat - 24.17%
2. tabby - 14.54%
3. Egyptian cat - 1.36%
4. diamondback - 0.57%
5. lynx - 0.27%

Image: https://images.unsplash.com/photo-1502877338535-766e1452684a
Inference time: 0.0884 sec
1. sports car - 29.37%
2. car wheel - 13.75%
3. beach wagon - 8.81%
4. grille - 1.40%
5. convertible - 1.35%

Image: https://images.unsplash.com/photo-1517336714731-489689fd1ca8
Inference time: 0.1642 sec
1. notebook - 49.61%
2. laptop - 10.80%
3. computer keyboard - 6.70%
4. space bar - 2.71%
5. iPod - 1.07%

Image: https://images.unsplash.com/photo-1493238792000-8113da705763
Inference time: 0.0947 sec
1. cab - 51.52%
2. sports car - 5.39%
3. parking meter - 2.82%
4. toaster - 1.61%
5.

Image 1 (dog)
The model confidently identified the “pug” class (99.17%). The image is clear and matches the training data well.

Image 2 (cat)
The low confidence (24.17% and 14.54%) is due to the fact that the classes are visually similar. The model distributes the probability between them.

Image 3 (car)
Moderate confidence (29.37%). The model partially focuses on individual parts (for example, a wheel), and not on the entire object.

Image 4 (laptop)
The prediction is quite accurate (49.61%), but there is confusion between similar categories (“laptop” and “notebook").

Image 5 (airplane)
Error: the model predicted “cab" (51.52%). The reason is a non—standard angle or background.

# TASK 3

In [9]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

for name, model in models_dict.items():
    total_time = 0

    for url in image_urls:
        _, t = classify_image(model, url)
        total_time += t

    avg_time = total_time / len(image_urls)

    print(f"\nModel: {name}")
    print(f"Avg inference time: {avg_time:.4f} sec")
    print(f"Parameters: {count_parameters(model)}")


Model: ResNet18
Avg inference time: 0.0411 sec
Parameters: 11689512

Model: ResNet50
Avg inference time: 0.0961 sec
Parameters: 25557032

Model: ResNet101
Avg inference time: 0.2107 sec
Parameters: 44549160


ResNet18 is the fastest but simplest.

ResNet50 — balance.

ResNet101 is the slowest, but more complex.

# TASK 4

In [10]:
error_urls = [
    "https://images.unsplash.com/photo-1542291026-7eec264c27ff",  # ambiguous
    "https://images.unsplash.com/photo-1519681393784-d120267933ba"  # unclear
]

for url in error_urls:
    print("\nImage:", url)

    results, _ = classify_image(models_dict["ResNet50"], url)

    print("Predicted:", results[0]["class"])


Image: https://images.unsplash.com/photo-1542291026-7eec264c27ff
Predicted: running shoe

Image: https://images.unsplash.com/photo-1519681393784-d120267933ba
Predicted: volcano


Errors occur due to the visual similarity of objects or non-standard images. The model focuses on the signs, not the meaning.

# EXTRA TASK

In [11]:
def batch_classify(model, urls):
    batch = torch.stack([preprocess(load_image(u)) for u in urls])

    start = time.time()

    with torch.no_grad():
        outputs = model(batch)

    return time.time() - start

batch_time = batch_classify(models_dict["ResNet50"], image_urls)

single_time = 0
for url in image_urls:
    _, t = classify_image(models_dict["ResNet50"], url)
    single_time += t

print(f"Batch time: {batch_time:.4f}")
print(f"Single total time: {single_time:.4f}")
print(f"Speedup: {single_time / batch_time:.2f}x")

Batch time: 0.3912
Single total time: 0.4794
Speedup: 1.23x


Batch processing is faster (1.23x) because it uses parallel computing.